# Large Neighbourhood Search (LNS) with Attention Model as Repair Operator

## Setup

In [2]:
from pathlib import Path

import torch

from core.lns.acceptance_criterion import AcceptanceCriterion, GreedyAcceptance
from core.lns.operators.destroy import TSPRandomSegmentDestroy
from core.lns.operators.repair.am_segment import AttentionModelRepair
from core.lns.search import search
from core.lns.utils import select_solution
from core.models.attention_model import AttentionModel
from core.models.attention_model.statistical_test import one_sided_paired_ttest
from core.models.attention_model.training import training_step
from core.models.attention_model.training_helpers import prepare_baseline, sync_baseline_
from core.models.utils import load_checkpoint, save_checkpoint
from core.problems.solution import BaseSolution
from core.problems.tsp import TSP, TSPSampler
from core.problems.tsp.solve import random_permutation


class DestroyRepairStep:
    def __init__(self, destroy, repair) -> None:
        self._destroy = destroy
        self._repair = repair

    def __call__(self, solution: BaseSolution, accept: AcceptanceCriterion) -> BaseSolution:
        partial = self._destroy(solution)
        candidate = self._repair(partial)
        mask = accept(solution, candidate)
        return select_solution(mask, candidate, solution)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NOTEBOOK_PATH = Path(__file__).parent if "__file__" in dir() else Path(".")

print(f"Using device: {DEVICE}")
print(f"Notebook path: {NOTEBOOK_PATH}")

Using device: cuda
Notebook path: .


In [3]:
%load_ext autoreload
%autoreload 2

Train the AM to be used as the repair operator in the LNS.

In [4]:
tsp_sampler = TSPSampler(
    TSP(
        num_locations=20,
        min_coordinate=0.0,
        max_coordinate=1.0,
        distribution_of_locations="uniform",
    ),
)

model = AttentionModel().to(DEVICE)

model_path = NOTEBOOK_PATH / "checkpoints" / "attention_model_for_repair.pth"
model_path.parent.mkdir(parents=True, exist_ok=True)

if model_path.exists():
    print(f"Loading model from {model_path}")
    load_checkpoint(path=model_path, model=model)
else:
    n_epochs = 15
    steps_per_epoch = 100
    batch_size = 512
    lr = 1e-4
    alpha = 0.05

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    baseline = prepare_baseline(model)

    for epoch in range(1, n_epochs + 1):
        for _ in range(steps_per_epoch):
            instance = tsp_sampler(batch_size).to(DEVICE)
            training_step(model=model, baseline=baseline, optimizer=optimizer, instance=instance)

        test = one_sided_paired_ttest(
            model,
            baseline,
            tsp_sampler,
            n_instances=1_000,
            batch_size=512,
            alpha=alpha,
            device=DEVICE,
        )
        if test.improved:
            sync_baseline_(baseline, model)
        print(
            f"Epoch {epoch:2d}: "
            f"obj_model={test.obj_model_mean:.4f}, "
            f"obj_baseline={test.obj_baseline_mean:.4f}, "
            f"updated={test.improved}"
        )

    save_checkpoint(
        path=model_path,
        model=model,
    )

Loading model from checkpoints\attention_model_for_repair.pth


## Basic LNS

In [5]:
experiments = [
    (20, 20, 1),
    (20, 10, 50),
    (20, 5, 100),
    (50, 15, 50),
    # (100, 15, 50),
]

results_of_experiments = []
for num_locations, destroy_size, n_iterations in experiments:
    print(
        f"Experiment: num_locations={num_locations}, destroy_size={destroy_size}, n_iterations={n_iterations}"
    )

    destroy = TSPRandomSegmentDestroy(destroy_size)
    repair = AttentionModelRepair(model)
    accept = GreedyAcceptance()

    sampler = TSPSampler(
        TSP(
            num_locations=num_locations,
            min_coordinate=0.0,
            max_coordinate=1.0,
            distribution_of_locations="uniform",
        ),
        seed=42,
    )
    batch = sampler(512).to(DEVICE)

    initial_solutions = random_permutation(batch)

    search_result = search(
        initial_solutions,
        improvement_step=DestroyRepairStep(destroy, repair),
        criterion=accept,
        n_iterations=n_iterations,
    )

    results_of_experiments.append(
        {
            "num_locations": num_locations,
            "destroy_size": destroy_size,
            "n_iterations": n_iterations,
            "mean_initial_cost": initial_solutions.objective().mean().item(),
            "mean_final_cost": search_result.best_solution.objective().mean().item(),
            "history": search_result.history,
        }
    )

Experiment: num_locations=20, destroy_size=20, n_iterations=1
Experiment: num_locations=20, destroy_size=10, n_iterations=50
Experiment: num_locations=20, destroy_size=5, n_iterations=100
Experiment: num_locations=50, destroy_size=15, n_iterations=50


In [6]:
results_of_experiments

[{'num_locations': 20,
  'destroy_size': 20,
  'n_iterations': 1,
  'mean_initial_cost': 10.428420066833496,
  'mean_final_cost': 6.219870567321777,
  'history': tensor([[11.1912, 11.2466, 10.2539,  ...,  9.1923, 10.6725, 10.6481],
          [ 6.3985,  6.4786,  6.0315,  ...,  5.3922,  7.2477,  6.9969]])},
 {'num_locations': 20,
  'destroy_size': 10,
  'n_iterations': 50,
  'mean_initial_cost': 10.47094440460205,
  'mean_final_cost': 4.91495418548584,
  'history': tensor([[12.1206, 11.4592,  9.5528,  ..., 11.2825, 10.3068, 11.1221],
          [ 9.3876,  9.6010,  7.5449,  ...,  8.8705,  9.1722,  8.3509],
          [ 8.1560,  8.5040,  7.5449,  ...,  7.0023,  9.1722,  7.3829],
          ...,
          [ 5.0107,  5.4681,  4.4371,  ...,  4.7418,  5.0893,  5.9501],
          [ 5.0107,  5.4681,  4.4371,  ...,  4.7418,  5.0893,  5.9501],
          [ 5.0107,  5.4681,  4.4371,  ...,  4.7418,  5.0893,  5.9501]])},
 {'num_locations': 20,
  'destroy_size': 5,
  'n_iterations': 100,
  'mean_initial_c

## Training the AM within the LNS loop

In [7]:
# TODO